# Imports

In [1]:
import xarray as xr
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import numpy as np
from scipy.stats import t
import dask

# PBSCLuster

In [2]:
cluster = PBSCluster(
    cores=4, # The number of cores you want
    memory='64GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='02:00:00', # Amount of wall time
)
cluster.scale(jobs=5)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.174:42853,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
# chunks='auto' appproaches this value
dask.config.set({'array.chunk-size': '8 GiB'})

# Function

In [4]:
def xr_regression(x, y, lag_x=0, lag_y=0, dim="time", alternative="two-sided"):
    """
    From https://stackoverflow.com/questions/52108417/how-to-apply-linear-regression-to-every-pixel-in-a-large-multi-dimensional-array
    requires scipy.stats as t
    Takes two xr.Datarrays of any dimensions (input data could be a 1D
    time series, or for example, have three dimensions e.g. time, lat,
    lon), and returns covariance, correlation, coefficient of
    determination, regression slope, intercept, p-value and standard
    error, and number of valid observations (n) between the two datasets
    along their aligned first dimension.

    Datasets can be provided in any order, but note that the regression
    slope and intercept will be calculated for y with respect to x.

    Inspired by:
    https://hrishichandanpurkar.blogspot.com/2017/09/vectorized-functions-for-correlation.html

    Parameters
    ----------
    x, y : xarray DataArray
        Two xarray DataArrays with any number of dimensions, both
        sharing the same first dimension
    lag_x, lag_y : int, optional
        Optional integers giving lag values to assign to either of the
        data, with lagx shifting x, and lagy shifting y with the
        specified lag amount.
    dim : str, optional
        An optional string giving the name of the dimension on which to
        align (and optionally lag) datasets. The default is 'time'.
    alternative : string, optional
        Defines the alternative hypothesis. Default is 'two-sided'.
        The following options are available:

        * 'two-sided': slope of the regression line is nonzero
        * 'less': slope of the regression line is less than zero
        * 'greater':  slope of the regression line is greater than zero

    Returns
    -------
    regression_ds : xarray.Dataset
        A dataset comparing the two input datasets along their aligned
        dimension, containing variables including covariance, correlation,
        coefficient of determination, regression slope, intercept,
        p-value and standard error, and number of valid observations (n).

    """

    # Shift x and y data if lags are specified
    if lag_x != 0:
        # If x lags y by 1, x must be shifted 1 step backwards. But as
        # the 'zero-th' value is nonexistant, xarray assigns it as
        # invalid (nan). Hence it needs to be dropped
        x = x.shift(**{dim: -lag_x}).dropna(dim=dim)

        # Next re-align the two datasets so that y adjusts to the
        # changed coordinates of x
        x, y = xr.align(x, y)

    if lag_y != 0:
        y = y.shift(**{dim: -lag_y}).dropna(dim=dim)

    # Ensure that the data are properly aligned to each other.
    x, y = xr.align(x, y)

    # Compute data length, mean and standard deviation along dim
    n = y.notnull().sum(dim=dim)
    xmean = x.mean(dim=dim)
    ymean = y.mean(dim=dim)
    xstd = x.std(dim=dim)
    ystd = y.std(dim=dim)

    # Compute covariance, correlation and coefficient of determination
    cov = ((x - xmean) * (y - ymean)).sum(dim=dim) / (n)
    cor = cov / (xstd * ystd)
    r2 = cor**2

    # Compute regression slope and intercept
    slope = cov / (xstd**2)
    intercept = ymean - xmean * slope

    # Compute t-statistics and standard error
    tstats = cor * np.sqrt(n - 2) / np.sqrt(1 - cor**2)
    stderr = slope / tstats

    # Calculate p-values for different alternative hypotheses.
    if alternative == "two-sided":
        pval = t.sf(np.abs(tstats), n - 2) * 2
    elif alternative == "greater":
        pval = t.sf(tstats, n - 2)
    elif alternative == "less":
        pval = t.cdf(np.abs(tstats), n - 2)

    # Wrap p-values into an xr.DataArray
    pval = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)

    # Combine into single dataset
    regression_ds = xr.merge(
        [
            cov.rename("cov").astype(np.float32),
            cor.rename("cor").astype(np.float32),
            r2.rename("r2").astype(np.float32),
            slope.rename("slope").astype(np.float32),
            intercept.rename("intercept").astype(np.float32),
            pval.rename("pvalue").astype(np.float32),
            stderr.rename("stderr").astype(np.float32),
            n.rename("n").astype(np.int16),
        ]
    )

    return regression_ds

# Data Import

In [5]:
EOF_ds = xr.open_dataset('/glade/work/acruz/E3SMv2LE/EANI_CANI_E3SMv2.nc', engine='h5netcdf')
EOF_ds

<xarray.Dataset> Size: 436kB
Dimensions:  (member: 21, time: 1212)
Coordinates:
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
    lev      (time) float64 10kB ...
    month    (time) int64 10kB ...
Data variables:
    EANI     (member, time) float64 204kB ...
    CANI     (member, time) float64 204kB ...
Attributes:
    title:        Atlantic Niño Index Timeseries
    description:  Contains EANI, CANI calculated from CESM2.1 LE2
    created:      2025-07-14

In [6]:
Uds = xr.open_dataset('/glade/work/acruz/E3SMv2LE/Uanom_tropic.nc', engine='h5netcdf')
Uds = Uds.chunk(chunks='auto')
Uds

<xarray.Dataset> Size: 2GB
Dimensions:  (member: 21, time: 1980, lev: 72, lon: 161)
Coordinates:
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
    month    (time) int64 16kB dask.array<chunksize=(1980,), meta=np.ndarray>
Dimensions without coordinates: member
Data variables:
    U        (member, time, lev, lon) float32 2GB dask.array<chunksize=(21, 1980, 72, 161), meta=np.ndarray>

In [7]:
Wds = xr.open_dataset('/glade/work/acruz/E3SMv2LE/OMEGAanom_tropic.nc', engine='h5netcdf')
Wds = Wds.chunk(chunks='auto')
Wds

<xarray.Dataset> Size: 2GB
Dimensions:  (member: 21, time: 1980, lev: 72, lon: 161)
Coordinates:
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
    month    (time) int64 16kB dask.array<chunksize=(1980,), meta=np.ndarray>
Dimensions without coordinates: member
Data variables:
    OMEGA    (member, time, lev, lon) float32 2GB dask.array<chunksize=(21, 1980, 72, 161), meta=np.ndarray>

# Select Data

In [8]:
dates = slice('1914-01-01', '2014-12-31')
Uds = Uds.sel(time=dates)
Wds = Wds.sel(time=dates)
EOF_ds = EOF_ds.sel(time=dates)
EOF_ds

<xarray.Dataset> Size: 436kB
Dimensions:  (member: 21, time: 1212)
Coordinates:
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
    lev      (time) float64 10kB ...
    month    (time) int64 10kB ...
Data variables:
    EANI     (member, time) float64 204kB ...
    CANI     (member, time) float64 204kB ...
Attributes:
    title:        Atlantic Niño Index Timeseries
    description:  Contains EANI, CANI calculated from CESM2.1 LE2
    created:      2025-07-14

# Seasonal Peak Mean


In [9]:
U_JJA = Uds['U'].sel(time=Uds.time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()
W_JJA = Wds['OMEGA'].sel(time=Wds.time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()

U_SON = Uds['U'].sel(time=Uds.time.dt.month.isin([9, 10, 11])).groupby('time.year').mean()
W_SON = Wds['OMEGA'].sel(time=Wds.time.dt.month.isin([9, 10, 11])).groupby('time.year').mean()

EANI = EOF_ds['EANI'].sel(time=EOF_ds['EANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()
CANI = EOF_ds['CANI'].sel(time=EOF_ds['CANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()

# Regression

In [10]:
U_EANI_JJAr = xr_regression(EANI, U_JJA, dim='year')
W_EANI_JJAr = xr_regression(EANI, W_JJA, dim='year')

In [11]:
U_CANI_JJAr = xr_regression(CANI, U_JJA, dim='year')
W_CANI_JJAr = xr_regression(CANI, W_JJA, dim='year')

In [12]:
U_EANI_SONr = xr_regression(EANI, U_SON, dim='year')
W_EANI_SONr = xr_regression(EANI, W_SON, dim='year')

In [13]:
U_CANI_SONr = xr_regression(CANI, U_SON, dim='year')
W_CANI_SONr = xr_regression(CANI, W_SON, dim='year')

# Export

In [14]:
U_EANI_JJAr

<xarray.Dataset> Size: 7MB
Dimensions:    (member: 21, lev: 72, lon: 161)
Coordinates:
  * member     (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * lev        (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lon        (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Data variables:
    cov        (member, lev, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    cor        (member, lev, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    r2         (member, lev, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    slope      (member, lev, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    intercept  (member, lev, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    pvalue     (member, lev, lon) float32 974kB 0.611 0.5581 ... 0.8001 0.6362
    stderr     (member, lev, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    n          (member, lev, lon) int16 487kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
Attributes:
    cell_measures:  area: area
    cell_methods:   time: mean
    long_name:      Zonal wind
    mdims:          1
    standard_name:  eastward_wind
    units:          m/s

In [15]:
regression = xr.Dataset(
    data_vars={
        "JJA_U_EANI_slope": (['member', 'level', 'lon'], U_EANI_JJAr['slope'].data),
        "JJA_U_EANI_pvalue":(['member', 'level', 'lon'], U_EANI_JJAr['pvalue'].data),
        "JJA_W_EANI_slope": (['member', 'level', 'lon'], W_EANI_JJAr['slope'].data),
        "JJA_W_EANI_pvalue":(['member', 'level', 'lon'], W_EANI_JJAr['pvalue'].data),
        "JJA_U_CANI_slope": (['member', 'level', 'lon'], U_CANI_JJAr['slope'].data),
        "JJA_U_CANI_pvalue":(['member', 'level', 'lon'], U_CANI_JJAr['pvalue'].data),
        "JJA_W_CANI_slope": (['member', 'level', 'lon'], W_CANI_JJAr['slope'].data),
        "JJA_W_CANI_pvalue":(['member', 'level', 'lon'], W_CANI_JJAr['pvalue'].data),
        "SON_U_EANI_slope": (['member', 'level', 'lon'], U_EANI_SONr['slope'].data),
        "SON_U_EANI_pvalue":(['member', 'level', 'lon'], U_EANI_SONr['pvalue'].data),
        "SON_W_EANI_slope": (['member', 'level', 'lon'], W_EANI_SONr['slope'].data),
        "SON_W_EANI_pvalue":(['member', 'level', 'lon'], W_EANI_SONr['pvalue'].data),
        "SON_U_CANI_slope": (['member', 'level', 'lon'], U_CANI_SONr['slope'].data),
        "SON_U_CANI_pvalue":(['member', 'level', 'lon'], U_CANI_SONr['pvalue'].data),
        "SON_W_CANI_slope": (['member', 'level', 'lon'], W_CANI_SONr['slope'].data),
        "SON_W_CANI_pvalue":(['member', 'level', 'lon'], W_CANI_SONr['pvalue'].data),
    },
   coords={
       "lon": U_EANI_JJAr['lon'].data,
       "level": U_EANI_JJAr['lev'].data
   },
    attrs={
        "Description": "Linear regression results from E3SMv2LE wind anomalies during JJA and SON against AN peak patterns in JJA from 1914 to 2014"
    }
)
regression

<xarray.Dataset> Size: 16MB
Dimensions:            (member: 21, level: 72, lon: 161)
Coordinates:
  * level              (level) float64 576B 0.1238 0.1828 0.2699 ... 993.8 998.5
  * lon                (lon) float64 1kB -180.0 -178.8 -177.5 ... 18.75 20.0
Dimensions without coordinates: member
Data variables: (12/16)
    JJA_U_EANI_slope   (member, level, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    JJA_U_EANI_pvalue  (member, level, lon) float32 974kB 0.611 ... 0.6362
    JJA_W_EANI_slope   (member, level, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    JJA_W_EANI_pvalue  (member, level, lon) float32 974kB 0.2848 ... 0.7331
    JJA_U_CANI_slope   (member, level, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    JJA_U_CANI_pvalue  (member, level, lon) float32 974kB 0.8194 ... 0.0005655
    ...                 ...
    SON_W_EANI_slope   (member, level, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    SON_W_EANI_pvalue  (member, level, lon) float32 974kB 0.7628 ... 0.9029
    SON_U_CANI_slope   (member, level, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    SON_U_CANI_pvalue  (member, level, lon) float32 974kB 0.1653 ... 0.3046
    SON_W_CANI_slope   (member, level, lon) float32 974kB dask.array<chunksize=(21, 72, 161), meta=np.ndarray>
    SON_W_CANI_pvalue  (member, level, lon) float32 974kB 0.5831 ... 0.4976
Attributes:
    Description:  Linear regression results from E3SMv2LE wind anomalies duri...

In [16]:
regression.to_zarr('/glade/work/acruz/E3SMv2LE/wind_regression', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [17]:
client.shutdown()